In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 日本語→英語翻訳モデル
model_name_ja_en = "Helsinki-NLP/opus-mt-ja-en"
tokenizer_ja_en = AutoTokenizer.from_pretrained(model_name_ja_en)
model_ja_en = AutoModelForSeq2SeqLM.from_pretrained(model_name_ja_en)

def translate_ja_en(text):
    """日本語→英語翻訳"""
    inputs = tokenizer_ja_en(text, return_tensors="pt")
    outputs = model_ja_en.generate(**inputs)
    return tokenizer_ja_en.decode(outputs[0], skip_special_tokens=True)

# 動作確認
test = "機械学習はコンピュータがデータから自動的に学習する技術です。"
print(translate_ja_en(test))

/home/qqqlq-desk/projects/nlp/.venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Machine learning is a technique in which computers automatically learn from data.


In [2]:
inputs = tokenizer_ja_en(test, return_tensors="pt")
print(type(inputs))
print(inputs.keys())
print(inputs)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
KeysView({'input_ids': tensor([[6124, 9988,   18, 9802,   34, 5250,  135, 7295, 1212, 9988,  154, 4021,
           74, 5832,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
{'input_ids': tensor([[6124, 9988,   18, 9802,   34, 5250,  135, 7295, 1212, 9988,  154, 4021,
           74, 5832,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [3]:
# 要約モデルをロードする
summarizer_name = "facebook/bart-large-cnn"
tokenizer_sum = AutoTokenizer.from_pretrained(summarizer_name)
model_sum = AutoModelForSeq2SeqLM.from_pretrained(summarizer_name)

def summarize(text, max_length=100, min_length=30):
    """英語テキストを要約する"""
    inputs = tokenizer_sum(text, return_tensors="pt", truncation=True)
    outputs = model_sum.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
    )
    return tokenizer_sum.decode(outputs[0], skip_special_tokens=True)

# 動作確認
test_en = "Machine learning is a technique in which computers automatically learn from data."
print(summarize(test_en))

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Machine learning is a technique in which computers automatically learn from data. Machine learning is used to teach computers how to use data to make better decisions.


In [4]:
# 長い日本語テキストで試す
long_text_ja = """
機械学習は人工知能の一分野であり、コンピュータがデータから自動的に学習し、
経験を通じて改善する能力を持つ技術です。従来のプログラミングでは、
人間がルールを明示的に記述する必要がありましたが、機械学習では
データからパターンを自動的に発見します。機械学習には大きく分けて、
教師あり学習、教師なし学習、強化学習の3種類があります。
教師あり学習はラベル付きデータを使って学習し、
教師なし学習はラベルなしデータからパターンを発見します。
強化学習は試行錯誤を通じて最適な行動を学習します。
近年では深層学習の発展により、画像認識、音声認識、自然言語処理など
様々な分野で人間を超える性能を達成しています。
"""

# パイプライン実行
print("=== 元の日本語テキスト ===")
print(long_text_ja.strip())

print("\n=== 英語に翻訳 ===")
translated = translate_ja_en(long_text_ja.strip())
print(translated)

print("\n=== 英語で要約 ===")
summarized = summarize(translated, max_length=80, min_length=20)
print(summarized)

=== 元の日本語テキスト ===
機械学習は人工知能の一分野であり、コンピュータがデータから自動的に学習し、
経験を通じて改善する能力を持つ技術です。従来のプログラミングでは、
人間がルールを明示的に記述する必要がありましたが、機械学習では
データからパターンを自動的に発見します。機械学習には大きく分けて、
教師あり学習、教師なし学習、強化学習の3種類があります。
教師あり学習はラベル付きデータを使って学習し、
教師なし学習はラベルなしデータからパターンを発見します。
強化学習は試行錯誤を通じて最適な行動を学習します。
近年では深層学習の発展により、画像認識、音声認識、自然言語処理など
様々な分野で人間を超える性能を達成しています。

=== 英語に翻訳 ===
Machine learning is a field of artificial intelligence, the technology that computers can automatically learn from data and improve through experience. In conventional programming, humans need to describe rules clearly, but in machine learning, there are three types of patterns that are automatically found from data: teacher learning, teacher learning, teacher learning, and reinforcement learning.

=== 英語で要約 ===
Machine learning is a field of artificial intelligence, the technology that computers can automatically learn from data. In conventional programming, humans need to describe rules clearly. In machine learning, there are three types of patterns tha

In [6]:
# configの中身を確認する
print(model_ja_en.config)

MarianConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 60715,
  "decoder_vocab_size": 60716,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "forced_eos_token_id": 0,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "marian",
  "normaliz

入力も出力も512トークンが上限なので、長い文章は途中で切れてしまいます。
解決策として文章を分割して翻訳

In [7]:
def translate_ja_en_chunked(text, chunk_size=3):
    """文章を句点で分割して翻訳する"""
    # 句点で分割する
    sentences = [s.strip() for s in text.split('。') if s.strip()]
    
    # chunk_sizeごとにまとめて翻訳する
    translated_chunks = []
    for i in range(0, len(sentences), chunk_size):
        chunk = '。'.join(sentences[i:i+chunk_size]) + '。'
        translated = translate_ja_en(chunk)
        translated_chunks.append(translated)
    
    return ' '.join(translated_chunks)

translated_chunked = translate_ja_en_chunked(long_text_ja.strip())
print("=== 分割翻訳 ===")
print(translated_chunked)

print("\n=== 要約 ===")
summarized2 = summarize(translated_chunked, max_length=80, min_length=20)
print(summarized2)

=== 分割翻訳 ===
Machine learning is a field of artificial intelligence, the technology that computers can automatically learn from data and improve through experience. In traditional programming, humans need to describe rules clearly, but in machine learning, they automatically find patterns. Teacher learning uses labeled data to learn patterns, and teacherless learning finds patterns in unlabeled data. Strong learning learns the best behavior through trial and error. In recent years, development of deep learning has made it more effective than humans in many fields, such as image recognition, audio recognition, and natural language processing.

=== 要約 ===
Machine learning is the technology that computers can automatically learn from data. Strong learning learns the best behavior through trial and error. Teacher learning uses labeled data to learn patterns, and teacherless learning finds patterns in unlabeled data.
